In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from utils import PSNR,SSIM
import torch




# ── 核心函数 ───────────────────────────────────────────────────────────────────
def fft_lowpass_filter(img: np.ndarray, c: float):
    """
    对图像做 FFT，仅保留 100c% 的低频成分，高频置零，再逆变换。

    Parameters
    ----------
    img : np.ndarray
        输入图像，形状 (H, W) 灰度 或 (H, W, 3) RGB，dtype uint8。
    c   : float
        保留低频的比例，取值范围 (0, 1]。
        例如 c=0.1 表示只保留 10% 的低频，90% 高频置零。

    Returns
    -------
    filtered : np.ndarray
        经低通滤波后的图像，dtype uint8，形状与 img 相同。
    """
    assert 0 < c <= 1, "c 必须在 (0, 1] 之间"

    def _filter_channel(channel: np.ndarray) -> np.ndarray:
        H, W = channel.shape

        # 1. FFT & 中心化
        f_shift = np.fft.fftshift(np.fft.fft2(channel))

        # 2. 构造低通掩膜（保留中心 c*H × c*W 的矩形区域）
        mask = np.zeros((H, W), dtype=np.float32)
        rH = int(round(H * c / 2))   # 半径（行方向）
        rW = int(round(W * c / 2))   # 半径（列方向）
        cH, cW = H // 2, W // 2      # 频谱中心
        mask[cH - rH : cH + rH, cW - rW : cW + rW] = 1.0

        # 3. 应用掩膜，逆变换
        filtered_channel = np.fft.ifft2(np.fft.ifftshift(f_shift * mask))
        return np.clip(np.abs(filtered_channel), 0, 255).astype(np.uint8)

    # 灰度 or 彩色分别处理
    if img.ndim == 2:
        filtered = _filter_channel(img)
    else:
        filtered = np.stack([_filter_channel(img[:, :, ch]) for ch in range(img.shape[2])], axis=-1)

    return filtered


# ── 可视化 + 评价 ──────────────────────────────────────────────────────────────
def visualize_and_evaluate(img: np.ndarray, c: float):
    """
    调用 fft_lowpass_filter，展示三张子图并打印 PSNR / SSIM。

    子图布局：
        [原图]  [低通滤波图]  [差值图(原图-滤波图)]
    """
    filtered = fft_lowpass_filter(img, c)

    # 差值图（绝对值，拉伸到 0‑255 方便观察）
    diff = np.abs(img.astype(np.int16) - filtered.astype(np.int16)).astype(np.uint8)

    # ── 指标 ──
    def to_tensor(img_np):
        return torch.from_numpy(img_np).float() / 255.0

    # 你的 fft 函数输出的是 uint8 numpy，转一下再传给 PSNR/SSIM
    # img_tensor      = to_tensor(img_array)       # 原图
    filtered_tensor = to_tensor(filtered)        # fft_lowpass_filter 的输出

    # psnr_val = psnr(torch.tensor(img), filtered_tensor)
    # ssim_val = ssim(torch.tensor(img), filtered_tensor)
   
    print(f"c = {c:.2f}  ({c*100:.1f}%)")
    # print(f"  PSNR : {psnr_val:.4f} dB")
    # print(f"  SSIM : {ssim_val:.4f}")

    # ── 绘图 ──
    cmap = "gray" if img.ndim == 2 else None
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].imshow(img,      cmap=cmap, vmin=0, vmax=255)
    axes[0].set_title("(Original)")
    axes[0].axis("off")

    axes[1].imshow(filtered, cmap=cmap, vmin=0, vmax=255)
    axes[1].set_title(f"(c={c:.2f})\n {c*100:.1f}% ")
    axes[1].axis("off")

    axes[2].imshow(diff,     cmap="hot", vmin=0, vmax=255)
    # axes[2].set_title(f"差值图 |原图 − 滤波图|\nPSNR={psnr_val:.2f} dB  SSIM={ssim_val:.4f}")
    axes[2].axis("off")

    plt.tight_layout()
    plt.show()
    
    return to_tensor(img).permute(2,0,1), filtered_tensor.permute(2,0,1)

# ── 使用示例 ───────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    from PIL import Image
    
    pnsr = PSNR()
    ssim = SSIM()


    idx = 801
    pic = Image.open(f"/home/liuy/data/raw/DIV2K/DIV2K_valid_HR/{idx:04d}.png").convert("RGB")
    pic = np.array(pic)
    img, filter = visualize_and_evaluate(pic, 0.8)
    print(pnsr(img, filter), ssim(img, filter))
    # print(img[0,0,0], filter[0,0,0])

In [ ]:
from utils import *
import torch

res_data = torch.load("/home/liuy/work/projects/NeuralOperatorSR/ppft_study/res_data/res_tensor.pt",weights_only=True)

idx = 200

res = res_data[idx, 0]
fft = torch.fft.fft2(res)
fft = torch.fft.fftshift(fft)
pf_h, pf_v = ppft2(res)

Eng = torch.abs(fft)
Engpp = torch.abs(pf_h)
print(pf_h.shape)

plt.imshow(Eng, cmap='hot')
plt.imshow(Engpp, cmap='hot')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn.functional as F
import torchvision.transforms.functional as TF
from utils.test_utils import fft_heatmap
from utils._utils import PSNR, SSIM

psnr = PSNR()
ssim = SSIM()
idx = 801
s = 2
patch_h_start = 0
patch_w_start = 0
patch_h = 512
patch_w = 600
# sheet = np.s_[:, patch_h_start:patch_h_start+patch_h, patch_w_start:patch_w_start+patch_w]
# sheet2 = np.s_[:, patch_h_start*s:patch_h_start*s+patch_h*s, patch_w_start*s:patch_w_start*s+patch_w*s]
gt_path = f"/home/liuy/data/raw/DIV2K/DIV2K_valid_HR/{idx:04d}.png"
lr_path = f"/home/liuy/data/raw/DIV2K/DIV2K_valid_LR_bicubic/X{s}/{idx:04d}x{s}.png"

gt = TF.to_tensor(TF.crop(Image.open(gt_path).convert("RGB"), top=patch_h_start*s, left=patch_w_start*s, height=patch_h*s, width=patch_w*s))
lr = TF.to_tensor(TF.crop(Image.open(lr_path).convert("RGB"), top=patch_h_start, left=patch_w_start, height=patch_h, width=patch_w)).unsqueeze(0)
# print(gt.shape, lr.shape)


bicubic = F.interpolate(lr, scale_factor=s, mode='bicubic', align_corners=False).squeeze(0)

_,_ = fft_heatmap(gt-bicubic, title="GT - Bicubic", )
print("Bicubic PSNR:", psnr(bicubic, gt))
print("Bicubic SSIM:", ssim(bicubic, gt))

In [ ]:
from torch.utils.tensorboard import SummaryWriter
import numpy as np
import matplotlib.pyplot as plt
from tensorboard.backend.event_processing import event_accumulator

from tensorboard.backend.event_processing import event_accumulator

save_path = "checkpoints/exp11/20260411_202723/events.out.tfevents.1775910443.jliu-SYS-420GP-TNR.1942798.0"
ea = event_accumulator.EventAccumulator(
    save_path,
    size_guidance={
        event_accumulator.SCALARS: 0,  # 0 表示加载全部
    }
)

ea.Reload()
print("Available tags:", ea.Tags())
def get_scalar(tag):
    events = ea.Scalars(tag)
    steps = np.array([e.step for e in events])
    values = np.array([e.value for e in events])
    return steps, values

# 取数据
steps_head_res, head_res = get_scalar("train/head_loss")
steps_train_loss, train_loss = get_scalar("train/train_loss")
steps_res_loss_freq, res_loss_freq = get_scalar("train/res_loss_freq")

steps_w1, w1 = get_scalar("train/w1")
steps_w2, w2 = get_scalar("train/w2")
steps_w3, w3 = get_scalar("train/w3")

steps_psnr, psnr_values = get_scalar("val/psnr")
steps_ssim, ssim_values = get_scalar("val/ssim")

# 作图
# ===== 美观优化版本 =====
plt.style.use('seaborn-v0_8-darkgrid')

fig, ax1 = plt.subplots(figsize=(8, 5), dpi=120)

# loss（统一风格：实线 + 不同颜色）
ax1.plot(steps_head_res, head_res, linewidth=2, label='head_loss')
ax1.plot(steps_res_loss_freq, res_loss_freq, linewidth=2, label='res_loss_freq')
ax1.plot(steps_train_loss, train_loss, linewidth=2, label='train_loss')
ax1.set_yscale('log')
ax1.set_ylabel("Loss (log scale)", fontsize=11)
ax1.set_xlabel("Step", fontsize=11)


# ax1.plot(steps_psnr, psnr_values, linewidth=2, label='PSNR')
# ax1.plot(steps_ssim, ssim_values, linewidth=2, label='SSIM')
# ax1.set_ylabel("Metric Value", fontsize=11)
# ax1.set_xlabel("Step", fontsize=11)



# 去掉上右边框
ax1.spines['top'].set_visible(False)

# 权重（统一风格：虚线 + 同色系）
ax2 = ax1.twinx()
ax2.plot(steps_w1, w1, linestyle='--', linewidth=2, alpha=0.8, label='w1')
ax2.plot(steps_w3, w3, linestyle='--', linewidth=2, alpha=0.8, label='w3')
ax2.plot(steps_w2, w2, linestyle='--', linewidth=2, alpha=0.8, label='w2')

ax2.set_ylabel("Weights", fontsize=11)
ax2.spines['top'].set_visible(False)

# 合并图例 + 放到外面
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(
    lines1 + lines2,
    labels1 + labels2,
    loc='upper center',
    bbox_to_anchor=(0.5, 1.15),
    ncol=3,
    frameon=False
)

plt.tight_layout()
plt.show()

